# Control dataset summary

This notebook summarizes the control candidates by control category, formula skeleton, space group, stability, theoretical status, and selected numerical properties.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/Users/andrewweiland/Andrews_Lab_Research/electride-ml-predictor')
DATA_PATH = PROJECT_ROOT / 'data/control/final_dataset/control_dataset.csv'
df = pd.read_csv(DATA_PATH)

# Give overlapping records one exclusive category for clean counts.
def category(control_type):
    if 'broad_background' in control_type:
        return 'Broad background'
    if 'isostructural_chemical' in control_type:
        return 'Isostructural chemical'
    return 'Same skeleton + space group'

df['control_category'] = df['control_type'].map(category)
for column in ['space_group', 'formula_skeleton', 'crystal_system']:
    df[column] = df[column].fillna('Not reported')

print(f'Structures: {len(df):,}')
print(f'Unique MP IDs: {df["mp_id"].nunique():,}')
print(f'Unique formula skeletons: {df["formula_skeleton"].nunique():,}')
print(f'Unique space groups: {df["space_group"].nunique():,}')
df.head()

In [ ]:
print('Control categories')
display(df['control_category'].value_counts().rename('structure_count').to_frame())

print('Most common formula skeletons')
display(df['formula_skeleton'].value_counts().head(25).rename('structure_count').to_frame())

print('Most common space groups')
display(df['space_group'].value_counts().head(25).rename('structure_count').to_frame())

print('Stability')
display(df['is_stable'].value_counts(dropna=False).rename('structure_count').to_frame())

print('Theoretical status')
display(df['theoretical'].value_counts(dropna=False).rename('structure_count').to_frame())

In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = ['#2f6690', '#d17a22', '#6a994e']

df['control_category'].value_counts().plot.bar(ax=axes[0, 0], color=colors)
axes[0, 0].set_title('Control category')
axes[0, 0].set_ylabel('Structures')
axes[0, 0].tick_params(axis='x', rotation=20)

df['formula_skeleton'].value_counts().head(20).sort_values().plot.barh(ax=axes[0, 1], color='#2f6690')
axes[0, 1].set_title('Top formula skeletons')
axes[0, 1].set_xlabel('Structures')

df['space_group'].value_counts().head(20).sort_values().plot.barh(ax=axes[1, 0], color='#6a994e')
axes[1, 0].set_title('Top space groups')
axes[1, 0].set_xlabel('Structures')

pd.crosstab(df['is_stable'].astype(str), df['theoretical'].astype(str)).plot.bar(ax=axes[1, 1], color=['#d17a22', '#2f6690'])
axes[1, 1].set_title('Stability by theoretical status')
axes[1, 1].set_xlabel('is_stable')
axes[1, 1].set_ylabel('Structures')
axes[1, 1].legend(title='theoretical')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df['is_stable'].astype(str).value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', startangle=90, colors=['#6a994e', '#d17a22'])
axes[0].set_title('Stability')
axes[0].set_ylabel('')

df['theoretical'].astype(str).value_counts().plot.pie(ax=axes[1], autopct='%1.1f%%', startangle=90, colors=['#2f6690', '#d17a22'])
axes[1].set_title('Theoretical status')
axes[1].set_ylabel('')

df['crystal_system'].value_counts().plot.bar(ax=axes[2], color='#6a994e')
axes[2].set_title('Crystal systems')
axes[2].set_ylabel('Structures')
axes[2].tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
numeric_columns = {
    'n_sites': 'Number of sites',
    'volume_A3': 'Volume (A^3)',
    'energy_above_hull_eV_per_atom': 'Energy above hull (eV/atom)',
    'band_gap_eV': 'Band gap (eV)',
}
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for axis, (column, title) in zip(axes.flat, numeric_columns.items()):
    values = pd.to_numeric(df[column], errors='coerce').dropna()
    axis.hist(values, bins=25, color='#2f6690', edgecolor='white')
    axis.set_title(title)
    axis.set_ylabel('Structures')
    axis.set_xlabel(title)
plt.tight_layout()
plt.show()

print('Numeric summary')
display(df[list(numeric_columns)].apply(pd.to_numeric, errors='coerce').describe().T)